In [1]:
from pathlib import Path
import pandas as pd
import seaborn as sns
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
import seaborn as sns  # statistical plotting
sns.set_style('darkgrid')  # consistent dark grid background for plots
import matplotlib.pyplot as plt  # plotting primitives
from sklearn.model_selection import train_test_split  # dataset splitting helper
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)  # evaluation metrics

from PIL import Image  # image I/O and basic manipulation

import cv2  # OpenCV for image processing
import warnings  # control warnings
from tqdm import tqdm  # progress bars

# Silence noisy warnings to keep notebook output clean
warnings.filterwarnings("ignore")

print('modules loaded')  # quick sanity check that imports succeeded
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers
from tensorflow.keras.applications import (
    EfficientNetB0,
    DenseNet121,
    VGG16,
    InceptionV3,
    MobileNetV2,
    ResNet50,
)
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

import cv2  # OpenCV for image processing
import warnings  # control warnings
from tqdm import tqdm  # progress bars

# Silence noisy warnings to keep notebook output clean
warnings.filterwarnings("ignore")

print('modules loaded')  # quick sanity check that imports succeeded

sns.set_style("darkgrid")

modules loaded
modules loaded


In [ ]:
TRAIN_DATASETS = {
    "dataset1": Path("dataset1_70_20_10"),
    "dataset2": Path("dataset2_70_20_10"),
}

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 1
LEARNING_RATE = 1e-4

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

In [3]:
def make_generators(dataset_root):
    train_gen = ImageDataGenerator(rescale=1./255)
    val_gen = ImageDataGenerator(rescale=1./255)
    test_gen = ImageDataGenerator(rescale=1./255)

    train_ds = train_gen.flow_from_directory(
        dataset_root / "train",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
    )

    val_ds = val_gen.flow_from_directory(
        dataset_root / "val",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
    )

    test_ds = test_gen.flow_from_directory(
        dataset_root / "test",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=False,
    )

    return train_ds, val_ds, test_ds

In [4]:
def build_vit(input_shape, num_classes):
    patch_size = 16
    projection_dim = 64
    transformer_layers = 6
    num_heads = 4
    dropout_rate = 0.1
    num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)

    def mlp(x, hidden_units):
        for units in hidden_units:
            x = layers.Dense(units, activation="gelu")(x)
            x = layers.Dropout(dropout_rate)(x)
        return x

    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(projection_dim, kernel_size=patch_size, strides=patch_size)(inputs)
    x = layers.Reshape((num_patches, projection_dim))(x)

    positions = tf.range(start=0, limit=num_patches, delta=1)
    pos_embed = layers.Embedding(num_patches, projection_dim)(positions)
    x = x + pos_embed

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization()(x)
        attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim
        )(x1, x1)

        x2 = layers.Add()([attention, x])

        x3 = layers.LayerNormalization()(x2)
        x3 = mlp(x3, [projection_dim * 2, projection_dim])

        x = layers.Add()([x3, x2])

    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs, outputs, name="VisionTransformer")

In [5]:
def build_resnet50(input_shape, num_classes):
    base = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="ResNet50")

In [6]:
def build_efficientnet(input_shape, num_classes):
    base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation="relu")(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="EfficientNetB0")

In [7]:
def build_mobilenet(input_shape, num_classes):
    base = MobileNetV2(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="MobileNetV2")

In [8]:
def build_inception(input_shape, num_classes):
    base = InceptionV3(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="InceptionV3")

In [9]:
def build_vgg16(input_shape, num_classes):
    base = VGG16(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="VGG16")

In [10]:
def build_densenet(input_shape, num_classes):
    base = DenseNet121(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="DenseNet121")

In [11]:
def build_efficientnet_variant(input_shape, num_classes):
    base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="EfficientNetB0_Variant")

In [12]:
MODEL_BUILDERS = {
    "VisionTransformer": build_vit,
    "ResNet50": build_resnet50,
    "EfficientNetB0": build_efficientnet,
    "MobileNetV2": build_mobilenet,
    "InceptionV3": build_inception,
    "VGG16": build_vgg16,
    "DenseNet121": build_densenet,
    "EfficientNetB0_Variant": build_efficientnet_variant,
}

In [ ]:
results = []

for model_name, builder in MODEL_BUILDERS.items():
    print("\n" + "=" * 80)
    print(f"TRAINING ARCHITECTURE: {model_name}")
    print("=" * 80)

    model_histories = {}

    for dataset_name, dataset_root in TRAIN_DATASETS.items():
        print(f"\nTraining {model_name} on {dataset_name}")

        train_ds, val_ds, test_ds = make_generators(dataset_root)

        input_shape = IMG_SIZE + (3,)
        num_classes = len(train_ds.class_indices)

        model = builder(input_shape, num_classes)

        model.compile(
            optimizer=Adam(LEARNING_RATE),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            verbose=1,
        )

        test_loss, test_acc = model.evaluate(test_ds, verbose=0)

        save_path = MODEL_DIR / f"{model_name}_{dataset_name}.h5"
        model.save(save_path)

        model_histories[dataset_name] = history.history

        results.append({
            "model": model_name,
            "dataset": dataset_name,
            "accuracy": test_acc,
            "loss": test_loss,
            "saved_path": str(save_path),
        })

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for dataset_name, hist in model_histories.items():
        axes[0].plot(hist["accuracy"], marker="o", label=f"{dataset_name} train")
        axes[0].plot(hist["val_accuracy"], linestyle="--", marker="x", label=f"{dataset_name} val")

    axes[0].set_title(f"{model_name} Accuracy Comparison")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True)

    for dataset_name, hist in model_histories.items():
        axes[1].plot(hist["loss"], marker="o", label=f"{dataset_name} train")
        axes[1].plot(hist["val_loss"], linestyle="--", marker="x", label=f"{dataset_name} val")

    axes[1].set_title(f"{model_name} Loss Comparison")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


TRAINING ARCHITECTURE: VisionTransformer

Training VisionTransformer on dataset1
Found 4200 images belonging to 4 classes.
Found 600 images belonging to 4 classes.
Found 1200 images belonging to 4 classes.
Epoch 1/5
 28/132 ━━━━━━━━━━━━━━━━━━━━ 2:22 1s/step - accuracy: 0.4115 - loss: 1.3779

In [ ]:
results_df = pd.DataFrame(results)
results_df.sort_values(["model", "accuracy"], ascending=[True, False])

In [ ]:
saved_models = list(MODEL_DIR.glob("*.h5"))
print(f"Total saved models: {len(saved_models)}")

for model_file in saved_models:
    print(model_file.name)